In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install opencv-python-headless


In [3]:
import os
import zipfile
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


In [4]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)


Dataset extracted to: /content/dataset


In [5]:
IMG_SIZE = 224

def load_images_from_folder(folder, label):
    X, y = [], []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(('.png','.jpg','.jpeg')):
                img_path = os.path.join(root, f)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = img.astype(np.float32) / 255.0
                X.append(img[..., np.newaxis])
                y.append(label)
    return X, y


In [6]:
X, y = [], []

# Old dataset
X_h, y_h = load_images_from_folder(
    "/content/dataset/Parkinson's Dataset Final/Hand Pd", 0
)
X_p, y_p = load_images_from_folder(
    "/content/dataset/Parkinson's Dataset Final/Hand Pd", 1
)

# New dataset
X_h2, y_h2 = load_images_from_folder(
    "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Healthy", 0
)
X_p2, y_p2 = load_images_from_folder(
    "/content/dataset/Parkinson's Dataset Final/New hand pd Dataset/Parkinson's", 1
)

X = np.array(X_h + X_p + X_h2 + X_p2)
y = np.array(y_h + y_p + y_h2 + y_p2)

print("Total images:", len(X))


Total images: 2128


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [8]:
def apply_clahe(X):
    out = []
    clahe = cv2.createCLAHE(2.0, (8,8))
    for img in X:
        g = (img.squeeze()*255).astype(np.uint8)
        g = clahe.apply(g)
        g = g.astype(np.float32)/255.0
        out.append(g[..., np.newaxis])
    return np.array(out)

X_train = apply_clahe(X_train)
X_test  = apply_clahe(X_test)


In [9]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1,0.1),
], name="vit_aug")


In [29]:
def mixup_tf(X, y, alpha=0.2):

    X = tf.convert_to_tensor(X)
    y = tf.convert_to_tensor(y)

    lam = tfp.distributions.Beta(alpha, alpha).sample([tf.shape(X)[0]])
    lam_x = tf.reshape(lam, (-1, 1, 1, 1))
    lam_y = tf.reshape(lam, (-1, 1))

    idx = tf.random.shuffle(tf.range(tf.shape(X)[0]))

    X2 = tf.gather(X, idx)
    y2 = tf.gather(y, idx)

    X_mix = lam_x * X + (1 - lam_x) * X2
    y_mix = lam_y * y + (1 - lam_y) * y2


    return X_mix.numpy(), y_mix.numpy()


In [25]:
!pip install tensorflow-probability
import tensorflow_probability as tfp


In [11]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=526):
        super().__init__()
        self.proj = layers.Conv2D(
            embed_dim, patch_size, patch_size
        )

    def call(self, x):
        x = self.proj(x)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))
        return x


In [12]:
class PositionalEncoding(layers.Layer):
    def __init__(self, n_patches, dim):
        super().__init__()
        self.pos = layers.Embedding(n_patches, dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1])
        return x + self.pos(positions)


In [13]:
class TransformerBlock(layers.Layer):
    def __init__(self, dim, heads=8, mlp_ratio=4, drop=0.1):
        super().__init__()
        self.attn = layers.MultiHeadAttention(heads, dim)
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.mlp = models.Sequential([
            layers.Dense(dim*mlp_ratio, activation="gelu"),
            layers.Dense(dim),
            layers.Dropout(drop)
        ])

    def call(self, x):
        x = self.norm1(x + self.attn(x,x))
        x = self.norm2(x + self.mlp(x))
        return x


In [17]:
def build_custom_vit():
    inputs = layers.Input((224,224,1))
    x = data_augmentation(inputs)


    x = PatchEmbedding(16, 526)(x)
    x = PositionalEncoding(196, 526)(x)


    channel_plan = [526, 256, 128, 64, 32, 16, 8, 4, 2, 1]

    for i in range(len(channel_plan) - 1):
        dim_in  = channel_plan[i]
        dim_out = channel_plan[i+1]


        x = TransformerBlock(dim_in)(x)


        x = layers.Dense(dim_out)(x)

    x = layers.LayerNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs, outputs)


In [18]:
lr = tf.keras.optimizers.schedules.CosineDecay(
    3e-4, decay_steps=3000, alpha=1e-2
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr, weight_decay=1e-4
)


In [21]:
model = build_custom_vit()

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),

    metrics=["accuracy"]
)

model.summary()


Model: "functional_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vit_aug (Sequential)            │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding_2               │ (None, None, 526)      │       135,182 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_2           │ (None, None, 526)      │       103,096 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_11            │ (None, None, 526)      │    11,084,924 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, None, 256)      │       134,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_12            │ (None, None, 256)      │     2,630,144 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, None, 128)      │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_13            │ (None, None, 128)      │       659,712 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, None, 64)       │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_14            │ (None, None, 64)       │       166,016 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, None, 32)       │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_15            │ (None, None, 32)       │        42,048 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, None, 16)       │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_16            │ (None, None, 16)       │        10,784 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_50 (Dense)                │ (None, None, 8)        │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_17            │ (None, None, 8)        │         2,832 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, None, 4)        │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_18            │ (None, None, 4)        │           776 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_56 (Dense)                │ (None, None, 2)        │            1

 Total params: 45,043,810 (171.83 MB)

 Trainable params: 15,014,603 (57.28 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 30,029,207 (114.55 MB)

In [22]:
for layer in model.layers:
    if isinstance(layer, PatchEmbedding):
        layer.trainable = False


In [31]:
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)
X_test  = np.asarray(X_test)
y_test  = np.asarray(y_test)

print(type(X_train), type(y_train))


<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [32]:
epochs = 50
batch_size = 16

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    if epoch < 8:
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=1,
            batch_size=batch_size,
            callbacks=[early_stop],
            shuffle=True,
            verbose=1
        )
    else:
        X_mix, y_mix = mixup_tf(X_train, y_train)
        X_mix = X_mix.numpy()
        y_mix = y_mix.numpy()

        history = model.fit(
            X_mix, y_mix,
            validation_data=(X_test, y_test),
            epochs=1,
            batch_size=batch_size,
            callbacks=[early_stop],
            shuffle=True,
            verbose=1
        )

    if early_stop.stopped_epoch > 0:
        print("Early stopping triggered.")
        break



Epoch 1/50


NotImplementedError: numpy() is only available when eager execution is enabled.

In [33]:
import numpy as np

X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
X_test  = np.asarray(X_test, dtype=np.float32)
y_test  = np.asarray(y_test, dtype=np.float32)

print(type(X_train), X_train.dtype)


<class 'numpy.ndarray'> float32


In [34]:
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(1024).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [35]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/8


NotImplementedError: numpy() is only available when eager execution is enabled.

In [36]:
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
    run_eagerly=True   # 🔑 THIS FIXES THE ERROR
)


In [37]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    verbose=1
)


Epoch 1/8


ValueError: Unknown variable: <Variable path=positional_encoding_2/embedding_2/embeddings, shape=(196, 526), dtype=float32, value=[[ 0.03315467 -0.01470015 -0.03644519 ... -0.00435973 -0.00453565
  -0.02256296]
 [ 0.04269571  0.03733155  0.00272297 ...  0.02657593 -0.04199129
  -0.01657797]
 [ 0.03757106 -0.00205482  0.04552123 ... -0.00255917  0.02365723
   0.04483664]
 ...
 [-0.04910197  0.00820084 -0.01535052 ...  0.00652629  0.00630755
   0.03485856]
 [-0.03615575  0.04968635  0.01658443 ... -0.00094431  0.01187062
   0.00573874]
 [-0.02897727  0.01232179 -0.00122434 ... -0.02736112 -0.02066383
  -0.00965058]]>. This optimizer can only be called for the variables it was originally built with. When working with a new set of variables, you should recreate a new optimizer instance.

In [38]:
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=1e-4
)


In [39]:
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
    run_eagerly=True   # keep eager ON for stability
)


In [40]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    verbose=1
)


Epoch 1/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.5149 - loss: 0.6931 - val_accuracy: 0.5117 - val_loss: 0.6931
Epoch 2/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 146s 1s/step - accuracy: 0.5101 - loss: 0.6931 - val_accuracy: 0.5117 - val_loss: 0.6930
Epoch 3/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 146s 1s/step - accuracy: 0.5043 - loss: 0.6932 - val_accuracy: 0.5117 - val_loss: 0.6930
Epoch 4/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.5208 - loss: 0.6928 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 5/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.4991 - loss: 0.6933 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 6/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 146s 1s/step - accuracy: 0.5176 - loss: 0.6928 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 7/8
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.5151 - loss: 0.6928 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 8/8
 13/107 ━━━━━━━━━━━━━━━━━━━━ 2:03 1s/step - accuracy: 0.4987 - loss: 0.6933

KeyboardInterrupt: 

In [41]:
for layer in model.layers:
    if isinstance(layer, PatchEmbedding) or isinstance(layer, PositionalEncoding):
        layer.trainable = False


In [42]:
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=1e-4,
    weight_decay=1e-4
)


In [43]:
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
    run_eagerly=True
)


In [44]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    verbose=1
)


Epoch 1/6
107/107 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.5188 - loss: 0.6927 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 2/6
107/107 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.5086 - loss: 0.6930 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 3/6
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.5030 - loss: 0.6932 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 4/6
107/107 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.5022 - loss: 0.6932 - val_accuracy: 0.5117 - val_loss: 0.6929
Epoch 5/6
 26/107 ━━━━━━━━━━━━━━━━━━━━ 1:48 1s/step - accuracy: 0.5737 - loss: 0.6908

KeyboardInterrupt: 